# Descargando los resultados electorales del 2026 de manera automatizada
**Autor: Nicolás Cardona - @cardonanl**

Este código fue creado para que cualquier persona con intereses en los resultados electorales de Colombia del 2026 pueda acceder a ellos a nivel de puesto de votación de manera automática.

**Elecciones disponibles:**
- Senado (`SE`)
- Cámara de Representantes (`CA`)
- Consultas Interpartidistas (`CN`)
- CITREP (`CT`)

**Niveles territoriales del nomenclator 2026:**
- Nivel 1 → País | Nivel 2 → Departamento | Nivel 3 → Municipio

> ⚠️ **Nota 2026:** Los códigos de departamento cambiaron respecto a 2023. Ya **no** son los códigos DANE de 2 dígitos (`76`). Ahora son códigos propios de 4 dígitos (ej: `3100` = Valle del Cauca). Ejecuta la celda 4b para ver todos los códigos disponibles.


---
## CELDA 1 — Configuración del usuario
### ⚠️ ESTE ES EL ÚNICO LUGAR DONDE EL USUARIO DEBE HACER CAMBIOS


In [ ]:
# =============================================================================
# PARÁMETROS DEL USUARIO — MODIFICAR AQUÍ
# =============================================================================

# --- TIPO DE ELECCIÓN ---
# Opciones disponibles para 2026:
#   'SE'  → Senado
#   'CA'  → Cámara de Representantes
#   'CN'  → Consultas Interpartidistas
#   'CT'  → CITREP
TIPO_ELECCION = 'SE'   # <--- CAMBIAR AQUÍ

# --- NIVEL TERRITORIAL ---
# Define el tipo de territorio por el que quieres filtrar.
# Opciones:
#   'departamento'  → filtra por código de departamento (ej: '3100' = Valle del Cauca)
#   'municipio'     → filtra por código de municipio (ej: '3100001' = Cali)
NIVEL_TERRITORIO = 'municipio'   # <--- CAMBIAR AQUÍ

# --- CÓDIGO DEL TERRITORIO ---
# ⚠️ IMPORTANTE: En 2026 los códigos de departamento son de 4 dígitos (NO los códigos DANE).
# Ejecuta la celda 4b para ver todos los códigos disponibles.
#
# Ejemplos de departamentos (4 dígitos):
#   '0100'=Antioquia  '0300'=Atlántico   '1600'=Bogotá D.C.  '0500'=Bolívar
#   '0700'=Boyacá     '0900'=Caldas      '1100'=Cauca        '1200'=Cesar
#   '1300'=Córdoba    '1500'=Cundinamarca '1900'=Huila        '2100'=Magdalena
#   '2300'=Nariño     '2500'=N.Santander  '2600'=Quindío      '2400'=Risaralda
#   '2700'=Santander  '2800'=Sucre        '2900'=Tolima       '3100'=Valle
#   '4000'=Arauca     '4400'=Caquetá      '4600'=Casanare     '4800'=La Guajira
#   '5000'=Guainía    '5200'=Meta         '5400'=Guaviare     '5600'=San Andrés
#   '6000'=Amazonas   '6400'=Putumayo     '6800'=Vaupés       '7200'=Vichada
#   '8800'=Consulados
CODIGO_TERRITORIO = '3100001'   # <--- CAMBIAR AQUÍ  (ejemplo: Cali)

# --- FILTRO OPCIONAL POR CANDIDATO ---
# Si quieres filtrar resultados para un solo candidato, ingresa su código (codcan).
# Déjalo en None para obtener todos los candidatos.
CODIGO_CANDIDATO = None   # <--- OPCIONAL

# --- MODO DE CARGA DEL NOMENCLATOR ---
# True  → usar archivo local (recomendado, menos carga al servidor)
# False → descargar desde la Registraduría en tiempo real
USAR_NOMENCLATOR_LOCAL = False   # <--- CAMBIAR si tienes el archivo local
RUTA_NOMENCLATOR_LOCAL = 'nomenclator_2026.json'  # <--- nombre del archivo local

# =============================================================================
print('Configuración cargada:')
print(f'  Tipo de elección : {TIPO_ELECCION}')
print(f'  Nivel territorial: {NIVEL_TERRITORIO}')
print(f'  Código territorio: {CODIGO_TERRITORIO}')
print(f'  Filtro candidato : {CODIGO_CANDIDATO if CODIGO_CANDIDATO else "Todos"}')
print(f'  Nomenclator local: {USAR_NOMENCLATOR_LOCAL}')


---
## CELDA 2 — Imports y constantes


In [ ]:
import requests
import pandas as pd
import json
import time
from pathlib import Path

# URL base de resultados 2026
BASE_URL = 'https://resultados.registraduria.gov.co'
NOMENCLATOR_URL = f'{BASE_URL}/json/nomenclator.json'
RESULTADOS_URL_TEMPLATE = BASE_URL + '/json/ACT/{tipo}/{codigo}.json'

# Mapeo de niveles del nomenclator 2026
NIVEL_NOMBRES = {
    1: 'País',
    2: 'Departamento',
    3: 'Municipio',
    4: 'Zona',
    5: 'Corregimiento',
    6: 'Puesto de votación',
}

# Mapeo sigla → número interno de elección en el nomenclator 2026
SIGLA_A_ELEC = {'SE': 1, 'CA': 2, 'CN': 6, 'CT': 7}

# Niveles a filtrar según el parámetro del usuario
NIVEL_FILTRO_MAP = {
    'departamento': 2,
    'municipio':    3,
    'puesto':       6,
}

# Renombrado de columnas del nomenclator para mayor claridad
# ⚠️ En 2026 el campo código es 'co' (no 'c' como en 2023)
COL_RENAME = {
    'i': 'id', 'n': 'nombre', 'co': 'codigo', 's': 'slug',
    'l': 'nivel', 'p': 'padre', 'r': 'refs', 'h': 'hijos'
}

# Headers para evitar bloqueo 403 del servidor de la Registraduría
HEADERS_BROWSER = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Referer': 'https://resultados.registraduria.gov.co/',
    'Accept': 'application/json, text/plain, */*',
}

print('Librerías importadas correctamente.')


---
## CELDA 3 — Cargar nomenclator
El nomenclator es el archivo maestro con todos los territorios y sus códigos únicos. Se puede cargar desde la web o desde un archivo local guardado previamente.


In [ ]:
def cargar_nomenclator(usar_local=False, ruta_local='nomenclator_2026.json'):
    """Carga el nomenclator desde archivo local o desde la Registraduría."""

    if usar_local:
        ruta = Path(ruta_local)
        if not ruta.exists():
            raise FileNotFoundError(
                f"No se encontró '{ruta_local}'. "
                "Descárgalo desde: https://resultados.registraduria.gov.co/json/nomenclator.json "
                "y colócalo en el mismo directorio que este notebook."
            )
        print(f'Cargando nomenclator desde archivo local: {ruta_local}')
        with open(ruta, 'r', encoding='utf-8') as f:
            data = json.load(f)
    else:
        print(f'Descargando nomenclator desde {NOMENCLATOR_URL} ...')
        response = requests.get(NOMENCLATOR_URL, headers=HEADERS_BROWSER, timeout=60)
        response.raise_for_status()
        data = response.json()
        with open('nomenclator_2026.json', 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False)
        print("Nomenclator guardado localmente como 'nomenclator_2026.json'")

    # Estructura 2026: data['amb'] es una lista de grupos por elección
    # Cada grupo: {'elec': N, 'ambitos': [...]}
    # Filtramos el grupo correspondiente al TIPO_ELECCION y aplanamos en dict por 'i'
    elec_num = SIGLA_A_ELEC.get(TIPO_ELECCION, 1)
    ambitos = {}
    for grupo in data.get('amb', []):
        if grupo.get('elec') == elec_num:
            for a in grupo.get('ambitos', []):
                ambitos[str(a['i'])] = a
            break

    print(f"Nomenclator cargado — {len(ambitos):,} ámbitos para '{TIPO_ELECCION}'")
    return data, ambitos


data, ambitos = cargar_nomenclator(
    usar_local=USAR_NOMENCLATOR_LOCAL,
    ruta_local=RUTA_NOMENCLATOR_LOCAL
)


---
## CELDA 4 — Explorar la estructura del nomenclator
Celda informativa: muestra qué niveles existen y cuántos ítems tiene cada uno.


In [ ]:
def explorar_nomenclator(ambitos):
    """Imprime un resumen de los niveles disponibles en el nomenclator."""
    from collections import Counter
    conteo = Counter(a.get('l') for a in ambitos.values())
    print(f'Total ámbitos: {len(ambitos):,}\n')
    for nivel in sorted(conteo):
        nombre = NIVEL_NOMBRES.get(nivel, f'Nivel {nivel}')
        ejemplo = next((a for a in ambitos.values() if a.get('l') == nivel), {})
        print(f"  Nivel {nivel} ({nombre}): {conteo[nivel]:>6,} ítems | ejemplo: co='{ejemplo.get('co','')}' n='{ejemplo.get('n','')}' ")

explorar_nomenclator(ambitos)


---
## CELDA 4b — Consultar códigos disponibles (departamentos y municipios)
Ejecuta estas celdas **antes** de configurar la CELDA 1. Te muestran todos los departamentos y municipios disponibles con sus códigos exactos.

> ⚠️ Los códigos de departamento en 2026 son de 4 dígitos (`'3100'` = Valle del Cauca), **no** los códigos DANE de 2 dígitos que se usaban en 2023.


In [ ]:
def listar_territorios(ambitos, nivel, filtro_prefijo=None):
    """
    Muestra los territorios disponibles para un nivel dado.

    nivel          : 2=Departamento, 3=Municipio
    filtro_prefijo : si se pasa (ej: '3100'), filtra solo ese departamento
    """
    filas = []
    vistos = set()
    for item in ambitos.values():
        if item.get('l') != nivel:
            continue
        codigo = str(item.get('co', ''))
        if filtro_prefijo and not codigo.startswith(filtro_prefijo):
            continue
        if codigo in vistos:
            continue
        vistos.add(codigo)
        filas.append({
            'codigo': codigo,
            'nombre': item.get('n', ''),
            'nivel':  nivel,
        })

    df = pd.DataFrame(filas).sort_values('codigo').reset_index(drop=True)
    return df


# --- Ver todos los DEPARTAMENTOS (nivel 2) ---
print('=' * 55)
print('DEPARTAMENTOS DISPONIBLES (nivel 2)')
print("Usa el 'codigo' en CODIGO_TERRITORIO con NIVEL_TERRITORIO='departamento'")
print('=' * 55)
df_deptos = listar_territorios(ambitos, nivel=2)
print(df_deptos.to_string(index=False))


In [ ]:
# --- Ver municipios de un departamento específico (nivel 3) ---
# Cambia 'codigo_depto' por el código de 4 dígitos del departamento que quieras
codigo_depto = '3100'   # <--- CAMBIAR AQUÍ  (ejemplo: Valle del Cauca)

print(f"MUNICIPIOS DEL DEPARTAMENTO '{codigo_depto}' (nivel 3)")
print("Usa el 'codigo' en CODIGO_TERRITORIO con NIVEL_TERRITORIO='municipio'")
print('=' * 60)
df_municipios = listar_territorios(ambitos, nivel=3, filtro_prefijo=codigo_depto)
print(df_municipios.to_string(index=False))


---
## CELDA 5 — Extraer territorios para el nivel elegido
Filtra los ítems del nomenclator según el nivel territorial y el código definidos en la celda de configuración.


In [ ]:
def extraer_puestos(ambitos, nivel_territorio, codigo_territorio):
    """
    Extrae los ítems del nomenclator para el territorio elegido.
    nivel_territorio : 'departamento', 'municipio' o 'puesto'
    codigo_territorio: prefijo del código territorial (ej: '3100' o '3100001')
    """
    nivel_objetivo = NIVEL_FILTRO_MAP.get(nivel_territorio)
    if nivel_objetivo is None:
        raise ValueError(f'nivel_territorio debe ser uno de: {list(NIVEL_FILTRO_MAP.keys())}')

    puestos = {}
    for item in ambitos.values():
        codigo = str(item.get('co', ''))
        nivel  = item.get('l', 0)
        if nivel == nivel_objetivo and codigo.startswith(codigo_territorio):
            puestos[codigo] = item

    resultado = list(puestos.values())
    print(f"Territorios encontrados para {nivel_territorio} '{codigo_territorio}': {len(resultado):,}")
    return resultado


puestos_unicos = extraer_puestos(ambitos, NIVEL_TERRITORIO, CODIGO_TERRITORIO)

# Vista previa
df_puestos = pd.DataFrame(puestos_unicos).rename(columns=COL_RENAME)
print(f'\nVista previa ({len(df_puestos)} registros):')
df_puestos.head(10)


---
## CELDA 6 — Extraer municipios para descarga (nivel 3)
La descarga de votos se hace a nivel de municipio. Esta celda obtiene los códigos de municipio dentro del territorio elegido.

> ℹ️ **Nota 2026:** El nomenclator ya no incluye nivel 6 (puestos físicos). La granularidad más fina disponible es nivel 3 (municipio).


In [ ]:
def extraer_municipios(ambitos, codigo_territorio):
    """
    Extrae los municipios (nivel 3) cuyo código empieza con el territorio elegido.
    En 2026 el nivel más granular del nomenclator es el municipio (nivel 3).
    """
    municipios = {}
    for item in ambitos.values():
        codigo = str(item.get('co', ''))
        nivel  = item.get('l', 0)
        if nivel == 3 and codigo.startswith(codigo_territorio):
            municipios[codigo] = item
    return list(municipios.values())


puestos_nivel3 = extraer_municipios(ambitos, CODIGO_TERRITORIO)
df_puestos3 = pd.DataFrame(puestos_nivel3).rename(columns=COL_RENAME)

print(f"Municipios para '{CODIGO_TERRITORIO}': {len(df_puestos3):,}")
df_puestos3.head(5)


---
## CELDA 7 — Descarga de resultados por municipio

Esta es la celda principal de extracción. Itera sobre todos los municipios y descarga los votos por candidato para el tipo de elección elegido.

⚠️ **Este proceso puede tardar mucho tiempo** según el número de municipios:
- Un municipio pequeño: segundos
- Un departamento completo: minutos a horas
- Senado nacional: varias horas

Se incluye un `time.sleep()` de cortesía para no sobrecargar el servidor.


In [ ]:
def descargar_votos(puestos, tipo_eleccion, codigo_candidato=None,
                    delay_seg=1.5, verbose_cada=50):
    """
    Descarga los votos para cada municipio.

    puestos          : lista de municipios del nomenclator (nivel 3)
    tipo_eleccion    : 'SE', 'CA', 'CN' o 'CT'
    codigo_candidato : si se especifica, filtra solo ese candidato (codcan)
    delay_seg        : segundos de espera entre requests (respetar al servidor)
    verbose_cada     : imprime progreso cada N municipios
    """
    codigos        = [str(p.get('co', '')) for p in puestos]
    nombres_puesto = {str(p.get('co', '')): p.get('n', 'Desconocido') for p in puestos}

    filas   = []
    errores = []

    for idx, codigo in enumerate(codigos):
        if idx % verbose_cada == 0:
            print(f'  [{idx}/{len(codigos)}] Procesando {codigo}...')

        url = RESULTADOS_URL_TEMPLATE.format(tipo=tipo_eleccion, codigo=codigo)

        try:
            resp = requests.get(url, headers=HEADERS_BROWSER, timeout=30)
            if resp.status_code != 200:
                errores.append({'codigo': codigo, 'status': resp.status_code})
                continue

            data_json = resp.json()

            for camara in data_json.get('camaras', []):
                for partido in camara.get('partotabla', []):
                    acto   = partido.get('act', {})
                    codpar = acto.get('codpar', '')
                    for candidato in acto.get('cantotabla', []):
                        if codigo_candidato and str(candidato.get('codcan', '')) != str(codigo_candidato):
                            continue
                        row = dict(candidato)
                        row['codpar']        = codpar
                        row['codigo_puesto'] = codigo
                        row['nombre_puesto'] = nombres_puesto.get(codigo, 'Desconocido')
                        filas.append(row)

            time.sleep(delay_seg)

        except requests.exceptions.RequestException as e:
            errores.append({'codigo': codigo, 'error': str(e)})
            continue

    if errores:
        print(f'\n⚠️  {len(errores)} municipios fallaron.')

    df = pd.DataFrame(filas)
    print(f'\nDescarga completa. Total filas: {len(df):,} | Municipios procesados: {len(codigos)}')
    return df, errores


# ⚠️ Ejecutar solo cuando estés listo — puede tardar mucho para ámbitos grandes
df_votos, lista_errores = descargar_votos(
    puestos=puestos_nivel3,
    tipo_eleccion=TIPO_ELECCION,
    codigo_candidato=CODIGO_CANDIDATO,
    delay_seg=1.5
)

df_votos.head(10)


---
## CELDA 8 — Enriquecer el DataFrame con nombres y jerarquía territorial
Añade columnas legibles usando el nomenclator: nombre del municipio, slug, código de departamento.


In [ ]:
def enriquecer_df(df, df_puestos3):
    """Añade columnas de nombre y jerarquía territorial al DataFrame de votos."""
    df = df.copy()

    mapeo_nombre = dict(zip(df_puestos3['codigo'].astype(str), df_puestos3['nombre']))
    mapeo_slug   = dict(zip(df_puestos3['codigo'].astype(str), df_puestos3['slug']))

    df['nombre_puesto'] = df['codigo_puesto'].map(mapeo_nombre).fillna('Desconocido')
    df['slug_puesto']   = df['codigo_puesto'].map(mapeo_slug).fillna('Desconocido')

    # Extraer jerarquía territorial desde el código del municipio
    # En 2026: códigos de departamento = 4 dígitos, municipio = 7 dígitos
    df['cod_depto']     = df['codigo_puesto'].astype(str).str[:4]
    df['cod_municipio'] = df['codigo_puesto'].astype(str).str[:7]

    return df


df_votos = enriquecer_df(df_votos, df_puestos3)
print('Columnas disponibles:', df_votos.columns.tolist())
df_votos.head(5)


---
## CELDA 9 — (Opcional) Agregar nombres de partidos
Diccionario de partidos conocidos a 2026. Actualizar con nuevos `codpar` que aparezcan.


In [ ]:
MAPEO_PARTIDOS = {
    '2':    'Partido Liberal',
    '3':    'Partido Conservador',
    '4':    'Cambio Radical',
    '5':    'Partido Verde',
    '9':    'Partido de la U',
    '12':   'Centro Democrático',
    '16':   'Colombia Renaciente',
    '18':   'Dignidad y Compromiso',
    '20':   'Nuevo Liberalismo',
    '21':   'Salvación Nacional',
    '22':   'Partido Verde Oxígeno',
    '24':   'Liga Gobernantes Anticorrupción',
    '26':   'Partido Ecologista Colombiano',
    '27':   'Fuerza de la Paz',
    '29':   'Nueva Fuerza Democrática',
    '31':   'Independientes',
    '33':   'Creemos',
    '35':   'Gente en Movimiento',
    '36':   'Fuerza Ciudadana',
    '37':   'Colombia Humana',
    '2484': 'Pacto Histórico Colombia Puede',
    '2677': 'Nuevo Liberalismo - Nueva Fuerza Democrática',
    '2717': 'Pacto Histórico',
    '2803': 'Pacto Histórico',
    '2997': 'Acuerdo de Coalición',
    '1773': 'Fuerza de la Paz - Mais',
}

if 'codpar' in df_votos.columns:
    df_votos['partido'] = df_votos['codpar'].astype(str).map(MAPEO_PARTIDOS).fillna('Desconocido')
    print('Partidos únicos encontrados:')
    print(df_votos[['codpar', 'partido']].drop_duplicates().sort_values('codpar').to_string(index=False))


---
## CELDA 10 — Exportar resultados


In [ ]:
nombre_archivo = f'resultados_{TIPO_ELECCION}_{NIVEL_TERRITORIO}_{CODIGO_TERRITORIO}_2026.csv'
df_votos.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
print(f'Archivo exportado: {nombre_archivo}')
print(f'Dimensiones: {df_votos.shape[0]:,} filas × {df_votos.shape[1]} columnas')
df_votos.describe(include='all')


---
## CELDA 11 — Utilidad: descarga de un solo municipio (modo rápido)
Útil para testear la conexión o extraer un municipio específico sin iterar todo el departamento.


In [ ]:
def descarga_municipio_unico(codigo_municipio, tipo_eleccion):
    """
    Descarga y retorna los votos de un único municipio.
    Ejemplo de uso: descarga_municipio_unico('3100001', 'SE')
    """
    url  = RESULTADOS_URL_TEMPLATE.format(tipo=tipo_eleccion, codigo=codigo_municipio)
    resp = requests.get(url, headers=HEADERS_BROWSER, timeout=30)
    resp.raise_for_status()
    data_json = resp.json()

    filas = []
    for camara in data_json.get('camaras', []):
        for partido in camara.get('partotabla', []):
            acto   = partido.get('act', {})
            codpar = acto.get('codpar', '')
            for candidato in acto.get('cantotabla', []):
                row = dict(candidato)
                row['codpar']        = codpar
                row['codigo_puesto'] = codigo_municipio
                filas.append(row)

    df = pd.DataFrame(filas)
    print(f'Municipio {codigo_municipio}: {len(df)} filas descargadas')
    return df


# Ejemplo de uso (descomentar para ejecutar):
# df_test = descarga_municipio_unico('3100001', TIPO_ELECCION)
# df_test.head()


---
## CELDA 12 — Utilidad: inspeccionar estructura JSON de la Registraduría
Para diagnosticar si la estructura del JSON cambió o verificar los campos disponibles.


In [ ]:
def inspeccionar_json_municipio(codigo_municipio, tipo_eleccion):
    """Muestra las claves de primer y segundo nivel del JSON de resultados."""
    url  = RESULTADOS_URL_TEMPLATE.format(tipo=tipo_eleccion, codigo=codigo_municipio)
    resp = requests.get(url, headers=HEADERS_BROWSER, timeout=30)
    resp.raise_for_status()
    data_json = resp.json()

    print('Claves top-level:', list(data_json.keys()))
    for camara in data_json.get('camaras', [])[:1]:
        print('\nClaves de camara:', list(camara.keys()))
        for partido in camara.get('partotabla', [])[:1]:
            print('Claves de partido:', list(partido.keys()))
            acto = partido.get('act', {})
            print('Claves de act:', list(acto.keys()))
            for candidato in acto.get('cantotabla', [])[:1]:
                print('Claves de candidato:', list(candidato.keys()))

# Ejemplo de uso (descomentar):
# inspeccionar_json_municipio('3100001', TIPO_ELECCION)
